# Verify Rewards

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from evidence_analysis import analyze,ROOT,read_json
analyze()
print(json.dumps(read_json(ROOT/'verification.json'),indent=2))
print('Verify Rewards definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Independent raw-event analysis definitions/execution completed.


{
  "status": "PASS",
  "evaluation_episodes": 30,
  "evaluation_decisions": 18000,
  "valid_pairs": 1170,
  "maximum_abs_errors": {
    "q": 2.220446049250313e-16,
    "reward": 2.220446049250313e-16,
    "pair": 0.0
  },
  "frozen_estimator_state_sha256": "12506cae1323efcfb65d723d2a5fa419387ac9b9fa5cfa74ef45b1e2272ac256",
  "fresh_worker_processes": 30,
  "random_action_sequences_exactly_replayed": true,
  "limits": [
    "TASK and MAX each have one trained policy; repeated episodes do not replicate training.",
    "Requested Unity seed diversity unproven; RANDOM diversity also changes action RNG.",
    "Success completion labels unavailable; explicit NA, natural endings and timeouts separately recorded."
  ]
}
Verify Rewards definitions/execution completed.
